In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Psychology Papers Collector with MongoDB - Updated for Content Extraction
Thu thập papers tâm lý học từ nhiều nguồn và lưu vào MongoDB với trích xuất nội dung PDF
"""

import os
import time
import json
import hashlib
import logging
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Optional
import xml.etree.ElementTree as ET
from urllib.parse import urljoin, urlparse
import re

# Third-party libraries
try:
    from scholarly import scholarly
    import arxiv
    from bs4 import BeautifulSoup
    import pandas as pd
    import requests
    from pymongo import MongoClient
    from bson import ObjectId
    import pymongo.errors
    # PDF processing libraries
    import PyPDF2
    import fitz  # PyMuPDF
    import pdfplumber
except ImportError as e:
    print(f"Cài đặt thư viện còn thiếu: {e}")
    print("Chạy: pip install scholarly arxiv beautifulsoup4 pandas requests pymongo PyPDF2 PyMuPDF pdfplumber")
    exit(1)

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class PsychologyPaperCollector:
    def __init__(self, 
                 output_dir="/home/aero/DoAnTotNghiep/Dataset/psychology_papers",
                 mongo_uri="mongodb://localhost:27017/",
                 db_name="psychology_papers"):
        """
        Initialize the collector with MongoDB
        """
        self.output_dir = Path(output_dir)
        self.pdf_dir = self.output_dir / "pdfs"
        
        # MongoDB setup
        self.mongo_uri = mongo_uri
        self.db_name = db_name
        self.client = None
        self.db = None
        self.collection = None
        
        # Create directories
        self.output_dir.mkdir(exist_ok=True)
        self.pdf_dir.mkdir(exist_ok=True)
        
        # Initialize MongoDB connection
        self._init_mongodb()
        
        # Psychology keywords for filtering
        self.psychology_keywords = [
            'psychology', 'psychological', 'psychologist', 'mental health',
            'cognitive', 'behavioral', 'behaviour', 'clinical psychology',
            'social psychology', 'developmental psychology', 'neuropsychology',
            'psychotherapy', 'psychiatry', 'psychological assessment',
            'psychological intervention', 'psychological treatment',
            'depression', 'anxiety', 'ptsd', 'schizophrenia', 'autism',
            'personality', 'emotion', 'motivation', 'learning', 'memory',
            'perception', 'attention', 'consciousness', 'stress'
        ]
        
        # Request session with headers
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })
    
    def _init_mongodb(self):
        """Initialize MongoDB connection and create indexes"""
        try:
            self.client = MongoClient(self.mongo_uri)
            self.db = self.client[self.db_name]
            self.collection = self.db.papers
            
            # Test connection
            self.client.admin.command('ping')
            logger.info(f"Connected to MongoDB: {self.db_name}")
            
            # Create indexes for better performance and uniqueness
            self.collection.create_index([("title", 1), ("authors", 1)], unique=True, sparse=True)
            self.collection.create_index([("doi", 1)], unique=True, sparse=True)
            self.collection.create_index([("source", 1)])
            self.collection.create_index([("publication_year", 1)])
            self.collection.create_index([("category", 1)])
            self.collection.create_index([("created_at", 1)])
            self.collection.create_index([("has_pdf", 1)])
            self.collection.create_index([("embedded", 1)])  # New index for embedding status
            
            # Text index for full-text search
            try:
                self.collection.create_index([
                    ("title", "text"),
                    ("abstract", "text"),
                    ("keywords", "text"),
                    ("content", "text")  # Include content field
                ])
            except pymongo.errors.OperationFailure:
                # Index might already exist
                pass
            
            logger.info("MongoDB indexes created successfully")
            
        except Exception as e:
            logger.error(f"Error connecting to MongoDB: {e}")
            raise
    
    def _extract_pdf_content(self, pdf_path: str) -> Optional[str]:
        """Extract text content from PDF file using multiple methods"""
        if not os.path.exists(pdf_path):
            logger.warning(f"PDF file not found: {pdf_path}")
            return None
        
        content = ""
        
        # Method 1: Try PyMuPDF (fitz) - usually best for academic papers
        try:
            doc = fitz.open(pdf_path)
            for page in doc:
                content += page.get_text()
            doc.close()
            
            if content.strip():
                logger.info(f"Successfully extracted content using PyMuPDF: {len(content)} characters")
                return content.strip()
                
        except Exception as e:
            logger.warning(f"PyMuPDF extraction failed for {pdf_path}: {e}")
        
        # Method 2: Try pdfplumber - good for structured documents
        try:
            with pdfplumber.open(pdf_path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        content += page_text + "\n"
            
            if content.strip():
                logger.info(f"Successfully extracted content using pdfplumber: {len(content)} characters")
                return content.strip()
                
        except Exception as e:
            logger.warning(f"pdfplumber extraction failed for {pdf_path}: {e}")
        
        # Method 3: Try PyPDF2 as fallback
        try:
            with open(pdf_path, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                for page in pdf_reader.pages:
                    content += page.extract_text() + "\n"
            
            if content.strip():
                logger.info(f"Successfully extracted content using PyPDF2: {len(content)} characters")
                return content.strip()
                
        except Exception as e:
            logger.warning(f"PyPDF2 extraction failed for {pdf_path}: {e}")
        
        logger.error(f"All PDF extraction methods failed for: {pdf_path}")
        return None
    
    def _is_psychology_related(self, title: str, abstract: str = "", keywords: str = "") -> bool:
        """Check if paper is psychology-related"""
        combined_text = f"{title} {abstract} {keywords}".lower()
        return any(keyword in combined_text for keyword in self.psychology_keywords)
    
    def _generate_hash(self, title: str, authors: str = "") -> str:
        """Generate hash for deduplication"""
        combined = f"{title}{authors}".encode('utf-8')
        return hashlib.md5(combined).hexdigest()
    
    def _save_paper_to_db(self, paper_data: Dict) -> bool:
        """Save paper to MongoDB with updated schema"""
        try:
            # Add metadata
            now = datetime.utcnow()
            paper_data['updated_at'] = now
            # Đừng thêm created_at vào paper_data ở đây!
            paper_data['hash'] = self._generate_hash(
                paper_data.get('title', ''), 
                paper_data.get('authors', '')
            )
            
            # Add new schema fields
            paper_data['embedded'] = False  # Not embedded yet
            
            # If PDF exists and was downloaded, extract content
            if paper_data.get('has_pdf', False) and paper_data.get('pdf_path'):
                content = self._extract_pdf_content(paper_data['pdf_path'])
                if content:
                    paper_data['content'] = content
                    paper_data['content_extracted'] = True
                    paper_data['content_length'] = len(content)
                else:
                    paper_data['content'] = ""
                    paper_data['content_extracted'] = False
                    paper_data['content_length'] = 0
            else:
                # No PDF available
                paper_data['content'] = ""
                paper_data['content_extracted'] = False
                paper_data['content_length'] = 0
            
            # Clean empty fields
            paper_data = {k: v for k, v in paper_data.items() if v is not None}
            
            # Use upsert to handle duplicates
            result = self.collection.update_one(
                {
                    "$or": [
                        {"title": paper_data.get('title'), "authors": paper_data.get('authors')},
                        {"doi": paper_data.get('doi')} if paper_data.get('doi') else {}
                    ]
                },
                {
                    "$set": paper_data,
                    "$setOnInsert": {"created_at": now}
                },
                upsert=True
            )
            
            if result.upserted_id:
                logger.info(f"New paper saved: {paper_data.get('title', '')[:50]}... (Content: {paper_data.get('content_length', 0)} chars)")
                return True
            else:
                logger.info(f"Paper updated: {paper_data.get('title', '')[:50]}... (Content: {paper_data.get('content_length', 0)} chars)")
                return False
                
        except pymongo.errors.DuplicateKeyError:
            logger.info(f"Duplicate paper skipped: {paper_data.get('title', '')[:50]}...")
            return False
        except Exception as e:
            logger.error(f"Error saving paper to MongoDB: {e}")
            return False
    
    def _download_pdf(self, pdf_url: str, title: str, source: str) -> Optional[str]:
        """Download PDF file"""
        if not pdf_url:
            return None
        
        try:
            # Clean filename
            safe_title = re.sub(r'[^\w\s-]', '', title)[:100]
            timestamp = int(time.time())
            filename = f"{source}_{safe_title}_{timestamp}.pdf"
            filepath = self.pdf_dir / filename
            
            # Download with timeout
            response = self.session.get(pdf_url, timeout=30, stream=True)
            response.raise_for_status()
            
            # Check if it's actually a PDF
            content_type = response.headers.get('content-type', '').lower()
            if 'application/pdf' not in content_type and 'pdf' not in content_type:
                logger.warning(f"Not a PDF file: {pdf_url}")
                return None
            
            # Save file
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            logger.info(f"Downloaded PDF: {filename}")
            return str(filepath.resolve())
            
        except Exception as e:
            logger.error(f"Error downloading PDF {pdf_url}: {e}")
            return None
    
    def collect_from_arxiv(self, query: str = "psychology", max_papers: int = 100) -> int:
        """Collect papers from arXiv"""
        logger.info(f"Collecting from arXiv: {query}")
        collected = 0
        
        try:
            search = arxiv.Search(
                query=f"cat:q-bio.NC OR all:{query}",  # Neuroscience and general search
                max_results=max_papers,
                sort_by=arxiv.SortCriterion.SubmittedDate
            )
            
            for paper in search.results():
                # Check if psychology-related
                if not self._is_psychology_related(paper.title, paper.summary):
                    continue
                
                # Extract data
                paper_data = {
                    'title': paper.title,
                    'authors': ', '.join([author.name for author in paper.authors]),
                    'abstract': paper.summary,
                    'keywords': ', '.join(paper.categories),
                    'publication_year': paper.published.year,
                    'journal': 'arXiv',
                    'doi': paper.doi,
                    'url': paper.entry_id,
                    'pdf_url': paper.pdf_url,
                    'source': 'arxiv',
                    'category': 'psychology',
                    'arxiv_id': paper.entry_id.split('/')[-1],
                    'published_date': paper.published,
                    'updated_date': paper.updated
                }
                
                # Download PDF
                pdf_path = self._download_pdf(paper.pdf_url, paper.title, 'arxiv')
                if pdf_path:
                    paper_data['pdf_path'] = pdf_path
                    paper_data['has_pdf'] = True
                else:
                    paper_data['has_pdf'] = False
                
                # Save to database (content extraction will happen in _save_paper_to_db)
                if self._save_paper_to_db(paper_data):
                    collected += 1
                
                time.sleep(1)  # Rate limiting
                
        except Exception as e:
            logger.error(f"Error collecting from arXiv: {e}")
        
        return collected
    
    def collect_from_pubmed(self, query: str = "psychology", max_papers: int = 100) -> int:
        """Collect papers from PubMed via NCBI API"""
        logger.info(f"Collecting from PubMed: {query}")
        collected = 0
        
        try:
            # Search for paper IDs
            search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
            search_params = {
                'db': 'pubmed',
                'term': f'{query}[Title/Abstract]',
                'retmax': max_papers,
                'retmode': 'xml'
            }
            
            response = self.session.get(search_url, params=search_params)
            response.raise_for_status()
            
            # Parse XML response
            root = ET.fromstring(response.content)
            id_list = root.find('IdList')
            
            if id_list is None:
                logger.warning("No papers found in PubMed")
                return 0
            
            paper_ids = [id_elem.text for id_elem in id_list.findall('Id')]
            
            # Fetch paper details
            fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
            batch_size = 20
            
            for i in range(0, len(paper_ids), batch_size):
                batch_ids = paper_ids[i:i+batch_size]
                
                fetch_params = {
                    'db': 'pubmed',
                    'id': ','.join(batch_ids),
                    'retmode': 'xml'
                }
                
                response = self.session.get(fetch_url, params=fetch_params)
                response.raise_for_status()
                
                # Parse papers
                root = ET.fromstring(response.content)
                articles = root.findall('.//PubmedArticle')
                
                for article in articles:
                    try:
                        # Extract paper info
                        title_elem = article.find('.//ArticleTitle')
                        abstract_elem = article.find('.//Abstract/AbstractText')
                        authors = article.findall('.//Author')
                        journal_elem = article.find('.//Journal/Title')
                        year_elem = article.find('.//PubDate/Year')
                        doi_elem = article.find('.//ELocationID[@EIdType="doi"]')
                        pmid_elem = article.find('.//PMID')
                        
                        if title_elem is None:
                            continue
                        
                        title = title_elem.text or ""
                        abstract = abstract_elem.text if abstract_elem is not None else ""
                        
                        # Check if psychology-related
                        if not self._is_psychology_related(title, abstract):
                            continue
                        
                        # Build author list
                        author_list = []
                        for author in authors:
                            lastname = author.find('.//LastName')
                            firstname = author.find('.//ForeName')
                            if lastname is not None:
                                name = lastname.text
                                if firstname is not None:
                                    name = f"{firstname.text} {name}"
                                author_list.append(name)
                        
                        paper_data = {
                            'title': title,
                            'authors': ', '.join(author_list),
                            'abstract': abstract,
                            'keywords': '',
                            'publication_year': int(year_elem.text) if year_elem is not None and year_elem.text else None,
                            'journal': journal_elem.text if journal_elem is not None else '',
                            'doi': doi_elem.text if doi_elem is not None else None,
                            'url': f"https://pubmed.ncbi.nlm.nih.gov/{pmid_elem.text}/" if pmid_elem is not None else '',
                            'source': 'pubmed',
                            'category': 'psychology',
                            'pmid': pmid_elem.text if pmid_elem is not None else None,
                            'has_pdf': False
                        }
                        
                        # Try to find PMC ID for potential PDF
                        pmc_elem = article.find('.//ELocationID[@EIdType="pmc"]')
                        if pmc_elem is not None:
                            paper_data['pmc_id'] = pmc_elem.text
                            paper_data['pdf_url'] = f"https://www.ncbi.nlm.nih.gov/pmc/articles/{pmc_elem.text}/pdf/"
                            pdf_path = self._download_pdf(paper_data['pdf_url'], title, 'pubmed')
                            if pdf_path:
                                paper_data['pdf_path'] = pdf_path
                                paper_data['has_pdf'] = True
                        
                        # Save to database
                        if self._save_paper_to_db(paper_data):
                            collected += 1
                        
                    except Exception as e:
                        logger.error(f"Error processing PubMed article: {e}")
                
                time.sleep(1)  # Rate limiting
                
        except Exception as e:
            logger.error(f"Error collecting from PubMed: {e}")
        
        return collected
    
    def collect_from_google_scholar(self, query: str = "psychology", max_papers: int = 50) -> int:
        """Collect papers from Google Scholar"""
        logger.info(f"Collecting from Google Scholar: {query}")
        collected = 0
        
        try:
            search_query = scholarly.search_pubs(query)
            
            for i, paper in enumerate(search_query):
                if i >= max_papers:
                    break
                
                try:
                    # Get paper details
                    paper_info = paper.get('bib', {})
                    title = paper_info.get('title', '')
                    abstract = paper_info.get('abstract', '')
                    
                    # Check if psychology-related
                    if not self._is_psychology_related(title, abstract):
                        continue
                    
                    # Extract PDF URL if available
                    pdf_url = paper.get('eprint_url', '')
                    
                    paper_data = {
                        'title': title,
                        'authors': ', '.join(paper_info.get('author', [])) if isinstance(paper_info.get('author'), list) else paper_info.get('author', ''),
                        'abstract': abstract,
                        'keywords': '',
                        'publication_year': int(paper_info.get('pub_year', 0)) if paper_info.get('pub_year') else None,
                        'journal': paper_info.get('venue', ''),
                        'url': paper.get('pub_url', ''),
                        'pdf_url': pdf_url,
                        'source': 'google_scholar',
                        'category': 'psychology',
                        'citation_count': paper.get('num_citations', 0),
                        'has_pdf': False
                    }
                    
                    # Try to download PDF
                    if pdf_url:
                        pdf_path = self._download_pdf(pdf_url, title, 'scholar')
                        if pdf_path:
                            paper_data['pdf_path'] = pdf_path
                            paper_data['has_pdf'] = True
                    
                    # Save to database
                    if self._save_paper_to_db(paper_data):
                        collected += 1
                    
                    time.sleep(2)  # Important: Rate limiting to avoid being blocked
                    
                except Exception as e:
                    logger.error(f"Error processing Scholar paper: {e}")
                    continue
                
        except Exception as e:
            logger.error(f"Error collecting from Google Scholar: {e}")
        
        return collected
    
    def extract_content_for_existing_pdfs(self):
        """Extract content for existing PDFs that don't have content yet"""
        logger.info("Extracting content for existing PDFs...")
        
        # Find papers with PDFs but no content
        query = {
            "has_pdf": True,
            "$or": [
                {"content": {"$exists": False}},
                {"content": ""},
                {"content_extracted": False}
            ]
        }
        
        papers_to_process = list(self.collection.find(query))
        logger.info(f"Found {len(papers_to_process)} papers to process")
        
        processed = 0
        for paper in papers_to_process:
            try:
                pdf_path = paper.get('pdf_path')
                if not pdf_path or not os.path.exists(pdf_path):
                    logger.warning(f"PDF not found for paper: {paper.get('title', '')[:50]}...")
                    continue
                
                content = self._extract_pdf_content(pdf_path)
                if content:
                    self.collection.update_one(
                        {"_id": paper["_id"]},
                        {
                            "$set": {
                                "content": content,
                                "content_extracted": True,
                                "content_length": len(content),
                                "updated_at": datetime.utcnow()
                            }
                        }
                    )
                    processed += 1
                    logger.info(f"Content extracted for: {paper.get('title', '')[:50]}... ({len(content)} chars)")
                else:
                    self.collection.update_one(
                        {"_id": paper["_id"]},
                        {
                            "$set": {
                                "content": "",
                                "content_extracted": False,
                                "content_length": 0,
                                "updated_at": datetime.utcnow()
                            }
                        }
                    )
                    logger.warning(f"Failed to extract content for: {paper.get('title', '')[:50]}...")
                
            except Exception as e:
                logger.error(f"Error processing paper {paper.get('title', '')[:50]}...: {e}")
        
        logger.info(f"Content extraction complete. Processed: {processed} papers")
        return processed
    
    def get_papers_for_embedding(self, limit: int = None) -> List[Dict]:
        """Get papers that have content but are not yet embedded"""
        query = {
            "content_extracted": True,
            "embedded": False,
            "content_length": {"$gt": 100}  # Only papers with substantial content
        }
        
        cursor = self.collection.find(query)
        if limit:
            cursor = cursor.limit(limit)
        
        papers = []
        for doc in cursor:
            doc['_id'] = str(doc['_id'])  # Convert ObjectId to string
            papers.append(doc)
        
        return papers
    
    def mark_as_embedded(self, paper_id: str):
        """Mark a paper as embedded"""
        try:
            from bson import ObjectId
            self.collection.update_one(
                {"_id": ObjectId(paper_id)},
                {
                    "$set": {
                        "embedded": True,
                        "embedded_at": datetime.utcnow(),
                        "updated_at": datetime.utcnow()
                    }
                }
            )
            return True
        except Exception as e:
            logger.error(f"Error marking paper as embedded: {e}")
            return False
    
    def collect_papers(self, queries: List[str] = None, max_papers_per_source: int = 50):
        """Main method to collect papers from all sources"""
        if queries is None:
            queries = [
                "cognitive psychology",
                "clinical psychology", 
                "social psychology",
                "developmental psychology",
                "neuropsychology",
                "psychological assessment",
                "mental health psychology",
                "behavioral psychology"
            ]
        
        total_collected = 0
        
        for query in queries:
            logger.info(f"Processing query: {query}")
            
            # Collect from each source
            arxiv_count = self.collect_from_arxiv(query, max_papers_per_source)
            pubmed_count = self.collect_from_pubmed(query, max_papers_per_source)
            scholar_count = self.collect_from_google_scholar(query, max_papers_per_source // 2)
            
            query_total = arxiv_count + pubmed_count + scholar_count
            total_collected += query_total
            
            logger.info(f"Query '{query}' results - arXiv: {arxiv_count}, PubMed: {pubmed_count}, Scholar: {scholar_count}")
            
            # Break between queries to be respectful
            time.sleep(5)
        
        # Extract content for any existing PDFs
        self.extract_content_for_existing_pdfs()
        
        logger.info(f"Collection complete. Total papers collected: {total_collected}")
        return total_collected
    
    def get_database_stats(self) -> Dict:
        """Get statistics about collected papers"""
        try:
            # Total papers
            total_papers = self.collection.count_documents({})
            
            # Papers with PDFs
            papers_with_pdf = self.collection.count_documents({"has_pdf": True})
            
            # Papers with extracted content
            papers_with_content = self.collection.count_documents({"content_extracted": True})
            
            # Papers already embedded
            papers_embedded = self.collection.count_documents({"embedded": True})
            
            # By source
            source_pipeline = [
                {"$group": {"_id": "$source", "count": {"$sum": 1}}},
                {"$sort": {"count": -1}}
            ]
            source_stats = list(self.collection.aggregate(source_pipeline))
            source_dict = {item['_id']: item['count'] for item in source_stats}
            
            # By year
            year_pipeline = [
                {"$match": {"publication_year": {"$ne": None, "$gt": 0}}},
                {"$group": {"_id": "$publication_year", "count": {"$sum": 1}}},
                {"$sort": {"_id": -1}},
                {"$limit": 10}
            ]
            year_stats = list(self.collection.aggregate(year_pipeline))
            year_dict = {item['_id']: item['count'] for item in year_stats}
            
            # Top journals
            journal_pipeline = [
                {"$match": {"journal": {"$ne": ""}}},
                {"$group": {"_id": "$journal", "count": {"$sum": 1}}},
                {"$sort": {"count": -1}},
                {"$limit": 10}
            ]
            journal_stats = list(self.collection.aggregate(journal_pipeline))
            journal_dict = {item['_id']: item['count'] for item in journal_stats}
            
            stats = {
                'total_papers': total_papers,
                'papers_with_pdf': papers_with_pdf,
                'papers_with_content': papers_with_content,
                'papers_embedded': papers_embedded,
                'ready_for_embedding': papers_with_content - papers_embedded,
                'pdf_percentage': (papers_with_pdf / total_papers * 100) if total_papers > 0 else 0,
                'content_percentage': (papers_with_content / total_papers * 100) if total_papers > 0 else 0,
                'embedding_percentage': (papers_embedded / total_papers * 100) if total_papers > 0 else 0,
                'by_source': source_dict,
                'recent_years': year_dict,
                'top_journals': journal_dict
            }
            
            return stats
            
        except Exception as e:
            logger.error(f"Error getting database stats: {e}")
            return {}
    
    def search_papers(self, 
                     text_query: str = None,
                     source: str = None,
                     year_range: tuple = None,
                     has_pdf: bool = None,
                     has_content: bool = None,
                     embedded: bool = None,
                     limit: int = 100) -> List[Dict]:
        """Search papers in the database"""
        try:
            query = {}
            
            # Text search
            if text_query:
                query["$text"] = {"$search": text_query}
            
            # Filter by source
            if source:
                query["source"] = source
            
            # Filter by year range
            if year_range:
                start_year, end_year = year_range
                query["publication_year"] = {"$gte": start_year, "$lte": end_year}
            
            # Filter by PDF availability
            if has_pdf is not None:
                query["has_pdf"] = has_pdf
            
            # Filter by content availability
            if has_content is not None:
                query["content_extracted"] = has_content
            
            # Filter by embedding status
            if embedded is not None:
                query["embedded"] = embedded
            
            # Execute query
            cursor = self.collection.find(query).limit(limit)
            
            # Convert to list and handle ObjectId
            results = []
            for doc in cursor:
                doc['_id'] = str(doc['_id'])  # Convert ObjectId to string
                results.append(doc)
            
            return results
            
        except Exception as e:
            logger.error(f"Error searching papers: {e}")
            return []
    
    def export_to_csv(self, filename: str = None) -> str:
        """Export papers to CSV file"""
        if filename is None:
            filename = f"psychology_papers_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        
        try:
            # Get all papers
            papers = list(self.collection.find({}))
            
            # Convert to DataFrame
            df = pd.DataFrame(papers)
            
            # Remove MongoDB ObjectId
            if '_id' in df.columns:
                df = df.drop('_id', axis=1)
            
            # Save to CSV
            csv_path = self.output_dir / filename
            df.to_csv(csv_path, index=False, encoding='utf-8')
            
            logger.info(f"Papers exported to: {csv_path}")
            return str(csv_path)
            
        except Exception as e:
            logger.error(f"Error exporting to CSV: {e}")
            return ""
    
    def close_connection(self):
        """Close MongoDB connection"""
        if self.client:
            self.client.close()
            logger.info("MongoDB connection closed")

def main():
    """Main execution function"""
    # MongoDB configuration
    MONGO_URI = "mongodb://localhost:27017/"  # Change this to your MongoDB URI
    DB_NAME = "psychology_papers"
    
    collector = PsychologyPaperCollector(
        mongo_uri=MONGO_URI,
        db_name=DB_NAME
    )
    
    try:
        # Define psychology-specific queries
        psychology_queries = [
            "cognitive psychology",
            "clinical psychology",
            "social psychology", 
            "developmental psychology",
            "personality psychology",
            "neuropsychology",
            "psychological assessment",
            "psychotherapy",
            "mental health",
            "psychological intervention", "trầm cảm", "depression", "rối loạn trầm cảm", "major depressive disorder",
    "triệu chứng trầm cảm", "depressive symptoms", "tự sát", "suicide", "ý nghĩ tự tử", "suicidal ideation",

    # Rối loạn lo âu / Anxiety Disorders
    "rối loạn lo âu", "anxiety disorder", "rối loạn lo âu tổng quát", "generalized anxiety disorder",
    "rối loạn hoảng sợ", "panic disorder", "rối loạn ám ảnh cưỡng chế", "obsessive-compulsive disorder",
    "rối loạn lo âu xã hội", "social anxiety disorder", "lo lắng", "anxiety", "sợ hãi", "fear",

    # Rối loạn lưỡng cực / Bipolar Disorder
    "rối loạn lưỡng cực", "bipolar disorder", "rối loạn cảm xúc lưỡng cực", "bipolar affective disorder",
    "hưng cảm", "mania",

    # Tâm thần phân liệt / Schizophrenia
    "tâm thần phân liệt", "schizophrenia", "ảo giác", "hallucination", "hoang tưởng", "delusion",

    # Rối loạn stress sau sang chấn / PTSD
    "rối loạn stress sau sang chấn", "post-traumatic stress disorder", "PTSD",
    "sang chấn tâm lý", "psychological trauma", "hồi tưởng", "flashback",

    # Rối loạn ăn uống / Eating Disorders
    "rối loạn ăn uống", "eating disorder", "chán ăn tâm thần", "anorexia nervosa", "cuồng ăn", "bulimia nervosa",

    # Rối loạn phát triển thần kinh / Neurodevelopmental Disorders
    "rối loạn phát triển thần kinh", "neurodevelopmental disorder", "rối loạn phổ tự kỷ", "autism spectrum disorder",
    "rối loạn tăng động giảm chú ý", "attention deficit hyperactivity disorder", "ADHD",
    "chậm phát triển trí tuệ", "intellectual disability",

    # Rối loạn nhân cách / Personality Disorders
    "rối loạn nhân cách", "personality disorder", "rối loạn nhân cách ranh giới", "borderline personality disorder",
    "rối loạn nhân cách chống đối xã hội", "antisocial personality disorder",
    "rối loạn nhân cách ám ảnh cưỡng chế", "obsessive-compulsive personality disorder",

    # Rối loạn ám ảnh sợ / Phobias
    "ám ảnh sợ xã hội", "social phobia", "ám ảnh sợ không gian kín", "claustrophobia",
    "ám ảnh sợ độ cao", "acrophobia",

    # Các vấn đề tâm lý khác / Other mental health issues
    "mất ngủ", "insomnia", "nghiện chất", "substance abuse", "rối loạn cảm xúc", "mood disorder",
    "rối loạn hành vi", "behavioral disorder", "tự kỷ", "autism", "tự làm hại bản thân", "self-harm"
        ]
        
        print("Starting Psychology Papers Collection with MongoDB...")
        print(f"Output directory: {collector.output_dir}")
        print(f"MongoDB: {MONGO_URI}/{DB_NAME}")
        
        # Collect papers
        total_collected = collector.collect_papers(psychology_queries, max_papers_per_source=30)
        
        # Show statistics
        stats = collector.get_database_stats()
        
        print("\n" + "="*50)
        print("COLLECTION SUMMARY")
        print("="*50)
        print(f"Total papers collected: {stats.get('total_papers', 0)}")
        print(f"Papers with PDF: {stats.get('papers_with_pdf', 0)} ({stats.get('pdf_percentage', 0):.1f}%)")
        
        print(f"\nBy source:")
        for source, count in stats.get('by_source', {}).items():
            print(f"  {source}: {count}")
        
        print(f"\nRecent years:")
        for year, count in stats.get('recent_years', {}).items():
            print(f"  {year}: {count}")
        
        print(f"\nTop journals:")
        for journal, count in list(stats.get('top_journals', {}).items())[:5]:
            print(f"  {journal}: {count}")
        
        print(f"\nPDFs location: {collector.pdf_dir}")
        
        # Export to CSV
        csv_file = collector.export_to_csv()
        if csv_file:
            print(f"Data exported to: {csv_file}")
        
    except KeyboardInterrupt:
        print("\nCollection interrupted by user")
    except Exception as e:
        logger.error(f"Error in main execution: {e}")
    finally:
        collector.close_connection()

if __name__ == "__main__":
    main()

2025-07-23 02:41:21,733 - INFO - Connected to MongoDB: psychology_papers
2025-07-23 02:41:21,737 - INFO - MongoDB indexes created successfully
2025-07-23 02:41:21,737 - INFO - Processing query: cognitive psychology
2025-07-23 02:41:21,738 - INFO - Collecting from arXiv: cognitive psychology
2025-07-23 02:41:21,738 - INFO - Requesting 30 results at offset 0
2025-07-23 02:41:21,738 - INFO - Requesting page of results


Starting Psychology Papers Collection with MongoDB...
Output directory: /home/aero/DoAnTotNghiep/Dataset/psychology_papers
MongoDB: mongodb://localhost:27017//psychology_papers


2025-07-23 02:41:22,691 - INFO - Sleeping for 2.999975 seconds
2025-07-23 02:41:25,691 - INFO - Requesting page of results
2025-07-23 02:41:26,021 - INFO - Sleeping for 2.999966 seconds
2025-07-23 02:41:29,022 - INFO - Requesting page of results
2025-07-23 02:41:29,343 - INFO - Sleeping for 2.999975 seconds
2025-07-23 02:41:32,344 - INFO - Requesting page of results
2025-07-23 02:41:32,670 - ERROR - Error collecting from arXiv: Page request resulted in HTTP 301: None (http://export.arxiv.org/api/query?search_query=cat%3Aq-bio.NC+OR+all%3Acognitive+psychology&id_list=&sortBy=submittedDate&sortOrder=descending&start=0&max_results=30)
2025-07-23 02:41:32,671 - INFO - Collecting from PubMed: cognitive psychology
2025-07-23 02:41:34,650 - INFO - Paper updated: The Eternal Allure of the Panacea: How Narratives ... (Content: 0 chars)
2025-07-23 02:41:34,652 - INFO - Paper updated: Language and/or memory: How to slice the domain-ca... (Content: 0 chars)
2025-07-23 02:41:34,653 - INFO - Paper u


COLLECTION SUMMARY
Total papers collected: 1031
Papers with PDF: 2 (0.2%)

By source:
  pubmed: 1031

Recent years:
  2026: 1
  2025: 956
  2024: 52
  2023: 9
  2022: 6
  2021: 3
  2020: 1
  2017: 1
  2011: 1

Top journals:
  Scientific reports: 38
  Cureus: 31
  Journal of affective disorders: 28
  Frontiers in psychiatry: 28
  Frontiers in psychology: 25

PDFs location: /home/aero/DoAnTotNghiep/Dataset/psychology_papers/pdfs
Data exported to: /home/aero/DoAnTotNghiep/Dataset/psychology_papers/psychology_papers_20250723_033237.csv


In [3]:
import os
import re
from pymongo import MongoClient
from datetime import datetime

# Thư viện trích xuất nội dung PDF (dùng PyMuPDF, pdfplumber, PyPDF2)
import fitz  # PyMuPDF
import pdfplumber
import PyPDF2

def normalize_title(title):
    title = title.lower()
    title = re.sub(r'[^\w\s]', '', title)
    title = re.sub(r'\s+', '_', title)
    return title

def extract_pdf_content(pdf_path):
    # Ưu tiên PyMuPDF
    try:
        doc = fitz.open(pdf_path)
        content = ""
        for page in doc:
            content += page.get_text()
        doc.close()
        if content.strip():
            return content.strip()
    except Exception:
        pass
    # Thử pdfplumber
    try:
        with pdfplumber.open(pdf_path) as pdf:
            content = ""
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    content += page_text + "\n"
        if content.strip():
            return content.strip()
    except Exception:
        pass
    # Thử PyPDF2
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            content = ""
            for page in pdf_reader.pages:
                page_text = page.extract_text()
                if page_text:
                    content += page_text + "\n"
        if content.strip():
            return content.strip()
    except Exception:
        pass
    return ""

PDF_DIR = "/home/aero/DoAnTotNghiep/Dataset/psychology_papers/pdfs"
MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "psychology_papers"
COLLECTION_NAME = "papers"

client = MongoClient(MONGO_URI)
collection = client[DB_NAME][COLLECTION_NAME]
def clean_filename(filename):
    # Loại bỏ tiền tố 'scholar_' nếu có
    if filename.lower().startswith('scholar_'):
        filename = filename[len('scholar_'):]
    # Loại bỏ hậu tố là chuỗi số (có thể có dấu gạch dưới trước số)
    filename = re.sub(r'_\d+$', '', filename)
    return filename
# Tạo map: normalized_title -> document _id
title_to_id = {}
for doc in collection.find({"title": {"$exists": True, "$ne": ""}}):
    norm_title = normalize_title(doc["title"])
    title_to_id[norm_title] = doc["_id"]

added = 0
for filename in os.listdir(PDF_DIR):
    if filename.lower().endswith(".pdf"):
        abs_path = os.path.join(PDF_DIR, filename)
        file_title = normalize_title(clean_filename(os.path.splitext(filename)[0]))
        matched = False
        for norm_title, doc_id in title_to_id.items():
            if norm_title in file_title or file_title in norm_title:
                # Trích xuất nội dung PDF
                content = extract_pdf_content(abs_path)
                content_extracted = bool(content)
                content_length = len(content)
                # Cập nhật vào database
                collection.update_one(
                    {"_id": doc_id},
                    {"$set": {
                        "pdf_path": abs_path,
                        "has_pdf": True,
                        "content": content,
                        "content_extracted": content_extracted,
                        "content_length": content_length,
                        "updated_at": datetime.utcnow()
                    }}
                )
                added += 1
                print(f"Gán file '{filename}' cho bài báo: '{norm_title}' | Content: {content_length} ký tự")
                matched = True
                break
        if not matched:
            print(f"Không tìm thấy bài báo phù hợp cho file: {filename}")

print(f"\nĐã gán {added} file PDF cho bài báo trong database và cập nhật nội dung.")
client.close()

Không tìm thấy bài báo phù hợp cho file: scholar_Personality psychology and economics_1753161176.pdf
Không tìm thấy bài báo phù hợp cho file: scholar_A European perspective on social anxiety disorder_1753162536.pdf
Không tìm thấy bài báo phù hợp cho file: scholar_Life-span theory in developmental psychology_1753161084.pdf
Không tìm thấy bài báo phù hợp cho file: scholar_Stress lo âu trầm cảm và một số yếu tố liên quan ở công nhân công ty may Phú Hưng tỉnh Hưng Yên năm _1753162868.pdf
Không tìm thấy bài báo phù hợp cho file: scholar_Does early psychological intervention promote recovery from posttraumatic stress_1753161620.pdf
Không tìm thấy bài báo phù hợp cho file: scholar_Toward a science of mental health_1753161515.pdf
Không tìm thấy bài báo phù hợp cho file: scholar_The neuropsychology of human memory_1753161250.pdf
Không tìm thấy bài báo phù hợp cho file: scholar_Thực trạng nguy cơ stress lo âu trầm cảm của học sinh trung học phổ thông huyện Yên Định Thanh Hóa_1753161665.pdf
Không

## Thông kê

In [3]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
collection = client["psychology_papers"]["papers"]

# Tổng số bài báo
total = collection.count_documents({})
print("Tổng số bài báo:", total)

# Số bài có file PDF
count = 0
for doc in collection.find({"has_pdf": True, "pdf_path": {"$exists": True, "$ne": ""}}):
    if os.path.exists(doc["pdf_path"]):
        count += 1

print("Số bài thực sự có file PDF:", count)

# Số bài theo nguồn
for src in collection.aggregate([
    {"$group": {"_id": "$source", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
]):
    print(f"Nguồn: {src['_id']}, Số bài: {src['count']}")

# Số bài theo năm
for year in collection.aggregate([
    {"$match": {"publication_year": {"$ne": None, "$gt": 0}}},
    {"$group": {"_id": "$publication_year", "count": {"$sum": 1}}},
    {"$sort": {"_id": -1}}
]):
    print(f"Năm: {year['_id']}, Số bài: {year['count']}")

Tổng số bài báo: 949
Số bài thực sự có file PDF: 0
Nguồn: pubmed, Số bài: 949
Năm: 2026, Số bài: 1
Năm: 2025, Số bài: 874
Năm: 2024, Số bài: 52
Năm: 2023, Số bài: 9
Năm: 2022, Số bài: 6
Năm: 2021, Số bài: 3
Năm: 2020, Số bài: 1
Năm: 2017, Số bài: 1
Năm: 2011, Số bài: 1


In [18]:
# Kết nối tới database
conn = sqlite3.connect('psychology_papers/papers.db')
cursor = conn.cursor()

# Liệt kê các bảng trong database
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Các bảng trong database:", tables)

# Giả sử bảng tên là 'papers', kiểm tra các trường trong bảng này
table_name = 'papers'  # Thay đổi nếu bảng của bạn có tên khác
cursor.execute(f"PRAGMA table_info({table_name});")
columns = cursor.fetchall()
print("Các trường trong bảng:", columns)

conn.close()

Các bảng trong database: [('papers',), ('sqlite_sequence',)]
Các trường trong bảng: [(0, 'id', 'INTEGER', 0, None, 1), (1, 'title', 'TEXT', 1, None, 0), (2, 'authors', 'TEXT', 0, None, 0), (3, 'abstract', 'TEXT', 0, None, 0), (4, 'keywords', 'TEXT', 0, None, 0), (5, 'publication_year', 'INTEGER', 0, None, 0), (6, 'journal', 'TEXT', 0, None, 0), (7, 'doi', 'TEXT', 0, None, 0), (8, 'url', 'TEXT', 0, None, 0), (9, 'pdf_url', 'TEXT', 0, None, 0), (10, 'pdf_path', 'TEXT', 0, None, 0), (11, 'source', 'TEXT', 0, None, 0), (12, 'category', 'TEXT', 0, None, 0), (13, 'created_at', 'TIMESTAMP', 0, 'CURRENT_TIMESTAMP', 0)]
